# 05 · LoRA: staged training of MedGemma on Pool 2 (Colab T4, arm: mixed)

## Settings (change ARM / STAGES_THIS_SESSION here)

In [ ]:
ARM = "mixed"                 # "mixed" (all), "emergency" (emergency only), "two_stage" (general first, then emergency)
CHUNK_SIZE = 30_000           # examples per stage (~5 h each on a T4: Gemma 3 trains in float32 there)
STAGES_THIS_SESSION = 2       # stages to attempt before stopping
SESSION_LIMIT_H = None        # save + stop cleanly after this many hours (Kaggle runs end at 12 h)
MAX_SEQ_LEN = 1024            # longer examples are cut to this many tokens

BASE_MODEL = "unsloth/medgemma-1.5-4b-it-unsloth-bnb-4bit"   # MedGemma 1.5 4B, 4-bit, speed-optimised copy
LORA_R, LORA_ALPHA = 16, 16
LR, BATCH, GRAD_ACC = 2e-4, 2, 8   # 16 examples per update
UPLOAD_EVERY_MIN = 30         # checkpoint this often, so a disconnect loses little
VAL_EXAMPLES = 600            # validation loss measured on this many held-out examples
SEED = 42

## Setup: Colab or Kaggle, install, where checkpoints live

In [ ]:
import os, sys, json, time, re, glob, random, hashlib, gc, shutil
SESSION_START = time.time()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
assert ON_COLAB or ON_KAGGLE, "Training needs a cloud GPU: Colab (T4) or Kaggle (GPU T4)"

!pip install -q unsloth
!pip install -q scikit-learn==1.9.1     # same split code as notebook 02

if ON_COLAB:
    # Google Drive keeps the checkpoints between sessions (Colab's own disk is wiped).
    # Colab asks you to allow Drive access the first time.
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = f"/content/drive/MyDrive/BN4601A_medgemma_lora/{ARM}"
    WORK = "/content/work"
else:
    # Kaggle keeps /kaggle/working as this run's saved output. To continue, attach the previous
    # run's output as an input: it then appears under /kaggle/input/ and is copied back here.
    CKPT_DIR = f"/kaggle/working/ckpt/{ARM}"
    WORK = "/tmp/work"                                  # raw data stays OUT of the saved output
    previous = sorted(glob.glob(f"/kaggle/input/**/ckpt/{ARM}/progress.json", recursive=True))
    if previous and not os.path.exists(f"{CKPT_DIR}/progress.json"):
        shutil.copytree(os.path.dirname(previous[-1]), CKPT_DIR)
        print("continuing from previous Kaggle run:", os.path.dirname(previous[-1]))
os.makedirs(CKPT_DIR, exist_ok=True)

# No Hugging Face login needed: the model copy and all datasets are public
from huggingface_hub import snapshot_download
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("checkpoints go to:", CKPT_DIR)

## Rebuild Pool 2 in the cloud (same code as notebook 02, so the same data)

In [ ]:
# Download the 4 raw datasets (about 680 MB, a few minutes)
for repo, patterns, name in [
    ("OpenMed/synthvision-validated-qwen-by-kimi", ["filtered_*.parquet"], "synthvision"),
    ("OpenMed/MedDialog", ["train.json", "validation.json"], "meddialog"),
    ("OpenMed/Medical-Reasoning-SFT-GPT-OSS-120B-Small", ["data/*.parquet"], "reasoning_sft"),
    ("GBaker/MedQA-USMLE-4-options", ["*.jsonl"], "medqa"),
]:
    snapshot_download(repo, repo_type="dataset", allow_patterns=patterns, local_dir=f"{WORK}/raw/{name}")

import pandas as pd
from pathlib import Path

In [ ]:
# --- copied from notebook 02 ---
import json, re, glob, random, hashlib, gc
from sklearn.model_selection import train_test_split

RAW = Path(f"{WORK}/raw")
SEED = 42
# No sampling here: every usable example goes into Pool 2.
# How many to TRAIN on is chosen later in the LoRA notebook (MAX_TRAIN_EXAMPLES).

# Words that make an example "emergency" (ED-like). Everything else is "general".
EMERGENCY_WORDS = [
    "emergency", "chest pain", "shortness of breath", "dyspnea", "breathless", "collapse", "syncope",
    "faint", "unconscious", "unresponsive", "seizure", "stroke", "slurred", "weakness on one side",
    "bleeding", "hemorrhage", "haemorrhage", "trauma", "injury", "fracture", "fall", "accident",
    "burn", "overdose", "poisoning", "anaphylaxis", "allergic reaction", "sepsis", "fever",
    "hypotension", "tachycardia", "cardiac arrest", "myocardial infarction", "heart attack",
    "acute abdomen", "severe pain", "vomiting blood", "hematemesis", "suicid", "confusion",
]
EMERGENCY_RE = re.compile(r"\b(" + "|".join(map(re.escape, EMERGENCY_WORDS)) + ")", re.I)

def scope(text):
    return "emergency" if EMERGENCY_RE.search(text or "") else "general"

def chat(user, assistant, system=None):
    """One training example in chat format: the same format ask() uses."""
    msgs = [{"role": "system", "content": system}] if system else []
    return msgs + [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]

def norm(text):
    return re.sub(r"[^a-z0-9]+", " ", (text or "").lower()).strip()

def as_text(messages):
    return " ".join(m["content"] or "" for m in messages)

In [ ]:
# --- copied from notebook 02 ---
def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

def medqa_prompt(q):
    options = "\n".join(f"{k}. {v}" for k, v in sorted(q["options"].items()))
    return f"{q['question']}\n\n{options}\n\nAnswer with the letter of the best option."

medqa_test = read_jsonl(RAW / "medqa/phrases_no_exclude_test.jsonl")
bench_idx = random.Random(SEED).sample(range(len(medqa_test)), 200)
bench_rows = [{"id": f"medqa_test_{i}", "prompt": medqa_prompt(medqa_test[i]),
               "answer": medqa_test[i]["answer_idx"], "answer_text": medqa_test[i]["answer"]} for i in bench_idx]

# One compiled pattern of all 200 questions (normalised start): finds hidden copies fast
BENCH_RE = re.compile("|".join(re.escape(norm(medqa_test[i]["question"])[:150]) for i in bench_idx))

def contains_bench(messages):
    return bool(BENCH_RE.search(norm(as_text(messages))))

print(len(bench_rows), "benchmark questions; example answer:", bench_rows[0]["answer"], "-", bench_rows[0]["answer_text"])

In [ ]:
# --- copied from notebook 02 ---
sv = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(RAW / "synthvision/filtered_*.parquet")))])
print("all records by source:", sv["source"].value_counts().to_dict())
sv = sv[sv["source"] != "multicare"].reset_index(drop=True)   # leakage guard: MultiCaRe = our pipeline test set
print("after dropping MultiCaRe:", len(sv))

SV_SYSTEM = "You are a clinical assistant. Answer using the imaging report provided; do not invent findings."

def sv_example(row):
    turns = json.loads(row["conversations"])
    msgs = [{"role": "system", "content": SV_SYSTEM}]
    for i, t in enumerate(turns):
        role = "user" if t["from"] == "human" else "assistant"
        text = t["value"]
        if i == 0:   # images aren't included, so the report text is the context
            text = f"Imaging report:\n{row['report']}\n\n{text}"
        msgs.append({"role": role, "content": text})
    return msgs

sv_all = pd.DataFrame({"id": "synthvision_" + sv["id"].astype(str), "source": "synthvision",
                       "scope": sv["report"].map(scope), "messages": sv.apply(sv_example, axis=1)})
del sv; gc.collect()
sv_all["scope"].value_counts()

In [ ]:
# --- copied from notebook 02 ---
parts = []
for name in ["train", "validation"]:
    d = pd.DataFrame(json.load(open(RAW / f"meddialog/{name}.json", encoding="utf-8")))
    d["id"] = f"meddialog_{name}_" + d.index.astype(str)
    parts.append(d)
md_raw = pd.concat(parts, ignore_index=True)
print("MedDialog total:", len(md_raw))
md_raw = md_raw[md_raw["doctor_response"].str.len().between(100, 2000)]      # drop empty / huge answers
print("answer 100-2,000 characters:", len(md_raw))

md_all = pd.DataFrame({"id": md_raw["id"].values, "source": "meddialog",
                       "scope": md_raw["patient_message"].map(scope).values,
                       "messages": [chat(q, a) for q, a in zip(md_raw["patient_message"], md_raw["doctor_response"])]})
del md_raw, parts; gc.collect()
md_all["scope"].value_counts()

In [ ]:
# --- copied from notebook 02 ---
parts = []
for f in sorted(glob.glob(str(RAW / "reasoning_sft/data/*.parquet"))):   # one file at a time to save memory
    rs = pd.read_parquet(f)
    question = rs["messages"].map(lambda m: m[0]["content"])
    answer = rs["messages"].map(lambda m: m[-1]["content"] or "")
    keep = ((rs["messages"].map(len) == 2)                        # single question + answer
            & answer.str.len().between(200, 3000)                 # not too long for training
            & ~question.str.contains("summarize the following", case=False))   # skip summarising chores
    tag = Path(f).stem.split("-")[1]                               # 00000 / 00001
    parts.append(pd.DataFrame({"id": [f"reasoning_{tag}_{i}" for i in rs.index[keep]], "source": "reasoning",
                               "scope": question[keep].map(scope).values,
                               "messages": [chat(q, a) for q, a in zip(question[keep], answer[keep])]}))
    print(Path(f).name, "->", int(keep.sum()), "usable of", len(rs))
    del rs, question, answer; gc.collect()
rs_all = pd.concat(parts, ignore_index=True)
rs_all["scope"].value_counts()

In [ ]:
# --- copied from notebook 02 ---
mq = pd.DataFrame(read_jsonl(RAW / "medqa/phrases_no_exclude_train.jsonl"))
mq_all = pd.DataFrame({"id": "medqa_train_" + mq.index.astype(str), "source": "medqa",
                       "scope": mq["question"].map(scope),
                       "messages": [chat(medqa_prompt(r), f"{r['answer_idx']}. {r['answer']}") for _, r in mq.iterrows()]})
mq_all["scope"].value_counts()

In [ ]:
# --- copied from notebook 02 ---
pool2 = pd.concat([sv_all, md_all, rs_all, mq_all], ignore_index=True)
del sv_all, md_all, rs_all, mq_all; gc.collect()
print("combined:", len(pool2))

# Leakage guard: drop any example that contains a MedQA benchmark question (about 2 minutes)
leaked = pool2["messages"].map(contains_bench)
print("examples containing a benchmark question (dropped):", int(leaked.sum()),
      pool2.loc[leaked, "source"].value_counts().to_dict())
pool2 = pool2[~leaked].reset_index(drop=True)

# Leakage guard: no exact duplicates (compare a short fingerprint of the normalised text)
fingerprint = pool2["messages"].map(lambda m: hashlib.md5(norm(as_text(m)).encode()).hexdigest())
dup = fingerprint.duplicated()
print("exact duplicates dropped:", int(dup.sum()), pool2.loc[dup, "source"].value_counts().to_dict())
pool2 = pool2[~dup].reset_index(drop=True)

# Split per source x scope, so every split has the same mix
pool2["split"] = "train"
for _, g in pool2.groupby(["source", "scope"]):
    _, rest = train_test_split(g.index, test_size=0.10, random_state=SEED)
    val, test = train_test_split(rest, test_size=0.50, random_state=SEED)
    pool2.loc[val, "split"] = "val"
    pool2.loc[test, "split"] = "test"

pd.crosstab([pool2["source"], pool2["scope"]], pool2["split"], margins=True)[["train", "val", "test", "All"]]

## Check: identical to notebook 02 (same examples, same order)

In [ ]:
EXPECTED = {   # fingerprints of the files notebook 02 saved on the laptop
    "train": (347779, "45eeb4f9e416a50a2709dccf2cefdcbf"),
    "val": (19322, "c964465d48c521c1db794599bd61cdde"),
    "test": (19324, "a4eefed2a76fcfd24dd6909d86291304"),
}
for split, (n, md5) in EXPECTED.items():
    ids = pool2.loc[pool2["split"] == split, "id"].tolist()
    got = (len(ids), hashlib.md5("\n".join(ids).encode()).hexdigest())
    assert got == (n, md5), f"{split} differs from notebook 02: {got}"
print("Pool 2 on Colab is identical to notebook 02")

## Progress: what's already done (read from Google Drive)

In [ ]:
progress = {"arm": ARM, "chunk": 0, "rows_done": 0, "history": [], "base_medqa_acc": None}
resume_dir = None
if os.path.exists(f"{CKPT_DIR}/progress.json") and os.path.exists(f"{CKPT_DIR}/adapter/adapter_model.safetensors"):
    resume_dir = CKPT_DIR
    progress = json.load(open(f"{CKPT_DIR}/progress.json"))
print(f"next: stage {progress['chunk'] + 1}, starting at example {progress['rows_done']:,}")
pd.DataFrame(progress["history"])

## Stage plan: shuffle once, cut into chunks (only this session's chunks kept in memory)

In [ ]:
train = pool2[pool2["split"] == "train"]
if ARM == "emergency":
    train = train[train["scope"] == "emergency"]
order = train.sample(frac=1, random_state=SEED)            # one fixed shuffle (seed 42)
if ARM == "two_stage":                                      # general first, emergency last
    order = pd.concat([order[order["scope"] == "general"], order[order["scope"] == "emergency"]])
starts = list(range(0, len(order), CHUNK_SIZE))
chunk_sizes = [min(CHUNK_SIZE, len(order) - s) for s in starts]

# Colab has ~12.7 GB RAM: keep only the chunks this session can reach, the rest stay empty
needed = range(progress["chunk"], min(progress["chunk"] + STAGES_THIS_SESSION, len(starts)))
chunks = [order.iloc[s:s + CHUNK_SIZE][["id", "source", "scope", "messages"]] if i in needed else None
          for i, s in enumerate(starts)]

# Fixed validation set: same examples every stage, every arm (so stages are comparable)
val_set = (pool2[pool2["split"] == "val"]
           .groupby("source", group_keys=False)
           .apply(lambda g: g.sample(VAL_EXAMPLES // 4, random_state=SEED)))
bench = pd.DataFrame(bench_rows)
del pool2, train, order; gc.collect()
print(f"{ARM}: {len(chunks)} stages of up to {CHUNK_SIZE:,} examples (this session: {list(needed)}); "
      f"val {len(val_set)}; MedQA bench {len(bench)}")

## Load MedGemma + LoRA (continues from the last checkpoint if there is one)

In [ ]:
from unsloth import FastModel, is_bf16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

model, tokenizer = FastModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,   # text only
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=SEED)
tokenizer = get_chat_template(tokenizer, "gemma-3")

if resume_dir:   # put the saved LoRA weights back in
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(f"{resume_dir}/adapter/adapter_model.safetensors"))
    print("continued from checkpoint")
model.print_trainable_parameters()

## Helpers: format examples, measure, save to Google Drive

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

def to_text(messages):
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    return tokenizer.apply_chat_template(msgs, tokenize=False).removeprefix("<bos>")

def as_dataset(df):
    return Dataset.from_dict({"text": [to_text(m) for m in df["messages"]]})

val_ds = as_dataset(val_set)

tok = getattr(tokenizer, "tokenizer", tokenizer)            # the text tokenizer inside the processor
LETTERS = ["A", "B", "C", "D"]
LETTER_IDS = [tok.encode(" " + l, add_special_tokens=False)[-1] for l in LETTERS]

def medqa_accuracy():
    """Share of the 200 held-out MedQA questions answered with the right letter.
    MedGemma writes its reasoning before the letter, so instead of generating text we end the
    prompt with "The best option is" and compare how likely " A", " B", " C", " D" come next."""
    FastModel.for_inference(model)
    right = 0
    for _, q in bench.iterrows():
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": q["prompt"]}],
                                               tokenize=False, add_generation_prompt=True) + "The best option is"
        ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        with torch.no_grad():
            logits = model(**ids).logits[0, -1]
        right += LETTERS[int(torch.argmax(logits[LETTER_IDS]))] == q["answer"]
    FastModel.for_training(model)
    return right / len(bench)

def save_checkpoint(note):
    """LoRA weights (~50-100 MB) + progress to Drive. Written to a temp name first, so a crash
    mid-save never leaves a half-written checkpoint."""
    import shutil
    tmp = f"{CKPT_DIR}/adapter_tmp"
    model.save_pretrained(tmp)
    shutil.rmtree(f"{CKPT_DIR}/adapter", ignore_errors=True)
    os.replace(tmp, f"{CKPT_DIR}/adapter")
    progress["last_note"] = note
    json.dump(progress, open(f"{CKPT_DIR}/progress_tmp.json", "w"), indent=1)
    os.replace(f"{CKPT_DIR}/progress_tmp.json", f"{CKPT_DIR}/progress.json")
    print(time.strftime("%H:%M"), "checkpoint saved:", note)

def over_time():
    return SESSION_LIMIT_H is not None and time.time() - SESSION_START > SESSION_LIMIT_H * 3600

class UploadEvery(TrainerCallback):
    """Every UPLOAD_EVERY_MIN minutes: save LoRA + how far we got in this stage.
    Near the session time limit: save and stop cleanly (the next session continues from here)."""
    def __init__(self, rows_at_start):
        self.rows_at_start, self.last, self.stopped = rows_at_start, time.time(), False
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.last > UPLOAD_EVERY_MIN * 60 or over_time():
            progress["rows_done"] = self.rows_at_start + state.global_step * BATCH * GRAD_ACC
            save_checkpoint(f"stage {progress['chunk'] + 1}: {progress['rows_done']:,} examples")
            self.last = time.time()
            if over_time():
                self.stopped, control.should_training_stop = True, True

if progress.get("medqa_method") != "letter_prob":   # first run, or scores from the old (buggy) text method
    with model.disable_adapter():                    # LoRA switched off = the untouched base model
        progress["base_medqa_acc"] = medqa_accuracy()
    for h in progress["history"]:                    # old per-stage scores are invalid; keep them labelled
        h["medqa_acc_old_method"] = h.pop("medqa_acc", None)
    if progress["history"]:                          # re-score the latest checkpoint with the fixed method
        progress["history"][-1]["medqa_acc"] = medqa_accuracy()
    progress["medqa_method"] = "letter_prob"
print("base MedGemma MedQA accuracy:", progress["base_medqa_acc"])

## Train: run the next stage(s)

In [ ]:
for _ in range(STAGES_THIS_SESSION):
    if progress["chunk"] >= len(chunks):
        print("all stages done"); break
    stage = progress["chunk"] + 1
    chunk = chunks[progress["chunk"]].iloc[progress["rows_done"]:]   # skip what an earlier session finished
    print(f"\n=== stage {stage}/{len(chunks)}: {len(chunk):,} examples ===")
    t0 = time.time()

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=as_dataset(chunk), eval_dataset=val_ds,
        args=SFTConfig(
            dataset_text_field="text", max_length=MAX_SEQ_LEN, num_train_epochs=1,
            per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
            per_device_eval_batch_size=4, learning_rate=LR, lr_scheduler_type="constant_with_warmup",
            warmup_steps=20, optim="adamw_8bit", weight_decay=0.01, seed=SEED,
            fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
            logging_steps=25, save_strategy="no", report_to="none", output_dir=f"{WORK}/out"),
    )
    # Learn only from the assistant's answers, not from copying the questions
    trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n",
                                      response_part="<start_of_turn>model\n")
    saver = UploadEvery(progress["rows_done"])
    trainer.add_callback(saver)
    trainer.train()
    if saver.stopped:   # session time limit: progress already saved mid-stage
        print("session time limit reached: saved, stopping. Run again to continue."); break

    # End of stage: measure, record, upload
    val_loss = trainer.evaluate()["eval_loss"]
    acc = medqa_accuracy()
    progress["history"].append({"stage": stage, "examples_seen": sum(chunk_sizes[:stage]),
                                "val_loss": round(val_loss, 4), "medqa_acc": acc,
                                "minutes": round((time.time() - t0) / 60)})
    progress["chunk"], progress["rows_done"] = progress["chunk"] + 1, 0
    save_checkpoint(f"stage {stage} done: val_loss {val_loss:.3f}, MedQA {acc:.1%}")
    del trainer; gc.collect(); torch.cuda.empty_cache()

pd.DataFrame(progress["history"])

## Learning curve

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(progress["history"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h["examples_seen"], h["val_loss"], marker="o"); ax[0].set(title="Validation loss (lower = better)", xlabel="training examples seen")
ax[1].plot(h["examples_seen"], h["medqa_acc"], marker="o", label="LoRA")
ax[1].axhline(progress["base_medqa_acc"], ls="--", color="grey", label="base MedGemma")
ax[1].set(title="MedQA (200 held-out questions)", xlabel="training examples seen"); ax[1].legend()
plt.tight_layout(); plt.show()